In [1]:
from collections import defaultdict
import json
from torch.utils.data import Dataset, DataLoader
import os, sys, torch, random
from pathlib import Path
import numpy as np
from torchvision import tv_tensors
from utils.utils import (
    crop_ultrasound_pil,
    extract_bbox_ultrasound_cv2,
    organ_to_class_dict,
    dataset_to_organ_dict,
    dataset_for_classification,
    dataset_for_segmentation,
    multi_cls_labels_dict,
    resize_pad,
)
from PIL import Image
from torchvision.transforms.v2.functional import pil_to_tensor, center_crop
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image
from tqdm import tqdm 
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
import numpy as np
from data_classes.datasets import USdatasetOmni

2026-01-23 16:48:06.030135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769183286.057001 2725151 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769183286.065256 2725151 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769183286.086968 2725151 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769183286.086985 2725151 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769183286.086988 2725151 computation_placer.cc:177] computation placer alr

In [2]:
from utils.utils import get_sft_transforms
from utils.paths import *


dataset = USdatasetOmni(
    base_dir=DATA_DIR,
    split="val_cls",
    transforms=get_sft_transforms(train=False),
    out_size=512,
    data_type="segmentation",
    ccl_crop=False,
    keep_aspect_ratio=True,
    self_norm=True, 
    include_testicles=True, 
    self_id=False,
    use_cluster_id = False,
    id_dropout=0.0
)

In [ ]:
from utils.stratified_splits import build_train_val_datasets


tr_idx, va_idx = stratified_80_20_indices(train_full, seed=seed)
# 3) Create Subset datasets (keep same transforms; you can add different aug for val if needed)
train_dataset = Subset(train_full, tr_idx)
# If you want different transforms for validation, rebuild a second base dataset w/ val transforms:
# val_base = USdatasetOmni(
#     DATA_DIR,
#     id_file_name,  # same source list, we'll index it with va_idx
#     transforms=get_sft_transforms(train=False) if get_sft_transforms_ is None else get_sft_transforms_(train=False),
#     data_type=args.dataset_type,
#     out_size=args.dataset_size,
#     ccl_crop=args.use_ccl_crop,
#     keep_aspect_ratio=args.keep_aspect_ratio,
#     include_testicles=True,
#     self_id = args.self_id,
#     num_clusters = args.num_clusters,
# )
val_base = copy.deepcopy(train_full)
val_base.aug = get_sft_transforms(train=False) if get_sft_transforms_ is None else get_sft_transforms_(train=False)
val_base.id_dropout = 0.0
val_dataset = Subset(val_base, va_idx)
print(f"[split] Total: {len(train_full)} | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [4]:
from argparse import Namespace
from collections import defaultdict
from copy import deepcopy
from transformers.trainer import Trainer
from transformers.training_args import TrainingArguments
from data_classes.datasets import USdatasetOmni
from torchvision.transforms import InterpolationMode, v2
import torch, wandb, random
from sklearn.metrics import accuracy_score
from nets.cls_net import OmniClsCBAM
from nets.segm_net import UNet2DFiLM, MedSAM, MedSAMPrompt
from utils.paths import DATA_DIR
from utils.utils import organ_to_class_dict, multi_cls_labels_dict, generate_run_hash
import numpy as np
from utils.utils import (
    get_sft_transforms,
    compute_dsc,
    class_to_organ_dict,
    compute_nsd,
    mask_overlap_visualization,
)
from utils.stratified_splits import build_train_val_datasets
from utils.paths import *
from torch.utils.data import Subset, ConcatDataset


In [5]:
training_args = TrainingArguments(
    output_dir="./loggings/debug",
    # num_train_epochs=args.epochs,
    max_steps=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    seed=42,
    save_strategy="steps",
    eval_strategy="steps",
    save_steps=int(1 / 100),
    eval_steps=int(1 / 100),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
    report_to=None,
    dataloader_num_workers=8,
    logging_steps=10,
    log_level="info",
    eval_accumulation_steps=100,
    optim="adamw_torch",
    learning_rate=1e-3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    max_grad_norm=1.0,
    gradient_accumulation_steps=2,
    # fp16=True,
    # push_to_hub=False,
)

In [ ]:
from safetensors.torch import load_file
from utils.distill_enc import apply_lora_to_encoder
peft_model_dict = torch.load("/media/raid0/US_FiLMUNet/loggings/peft_unet4_2/final_model_lora.pth")
model = UNet2DFiLM(
    in_channels=3,
    num_classes=1,
    # n_organs=len(organ_to_class_dict),
    n_organs=10,
    size=32,
    depth=4,
    film_start=0,
    use_film=True,
    film_embed=64,
    film_autoembed = True,
    distill=False
)
peft_model = apply_lora_to_encoder(model, lora_r=128, lora_alpha=16, lora_dropout=0.1)
peft_model.load_state_dict(peft_model_dict)

In [ ]:
merged = peft_model.merge_and_unload()
torch.save(merged.state_dict(), "/media/raid0/US_FiLMUNet/loggings/peft_unet4_2/merged_unet.pth")

In [ ]:
from train.segm_train import UNet2DFiLM
model = UNet2DFiLM(
    in_channels=3,
    num_classes=1,
    # n_organs=len(organ_to_class_dict),
    n_organs=9,
    size=32,
    depth=4,
    film_start=0,
    use_film=True,
    film_embed=64,
    film_autoembed = True,
    distill=False
)
model.cuda()
print(sum([p.numel() for p in model.parameters()]))
from safetensors.torch import load_file

state_dict = load_file("/work/tesi_nmorelli/UUSIC_new/checkpoints/unet4/model.safetensors")
model.load_state_dict(state_dict)

In [7]:
from nets.unet_attn import UNet2DAttn

model = UNet2DAttn(
    in_channels=3,
    num_classes=1,
    n_organs=10,
    size=32,
    depth=5,
    attn_start=0,      # Start attention from first level
    use_attn=True,     # Enable attention
    img_size=512,      # Input image size
    patch_size=8,     # 16×16 patches → 256 patches total
    emb_dim=768,       # Embedding dimension
    n_heads=8,          # Number of attention heads,
    # n_transformer_layers = 12,
    distill = 0
)
model.cuda()
print(sum([p.numel() for n, p in model.named_parameters() if 'distill_model' not in n]))


119799105


In [ ]:
sd = model.state_dict()
seen = {}
for k, v in sd.items():
    ptr = v.storage().data_ptr() if v.is_cuda is False else v.untyped_storage().data_ptr()
    seen.setdefault(ptr, []).append(k)

shared = {ptr:names for ptr,names in seen.items() if len(names) > 1}
print("num shared storages:", len(shared))
for ptr, names in list(shared.items())[:10]:
    print(names)


In [ ]:
from torch import nn
import torch

updated = 0
with torch.no_grad():
    for name, module in model.named_modules():
        # your FiLM2d has attribute .embed which is nn.Embedding
        # print(name)
        if hasattr(module, "patch_embed"):
            #  and isinstance(getattr(module, "embed"), nn.Embedding)
            # emb: nn.Embedding = module.embed
            # W = emb.weight  # [num_embeddings, emb_dim]
            # if W.shape[0] < 10:
            #     raise ValueError(f"{name}.embed has {W.shape[0]} rows (<10); can't set rows 8 and 9.")
            # mean_0_7 = W[0:8].mean(dim=0)   # [emb_dim]
            # W[8].copy_(mean_0_7)
            # W[9].copy_(mean_0_7)
            updated += 1
print(updated)

In [ ]:
data["masks"].shape

In [ ]:
data = dataset.__getitem__(0)
data.keys()
model.eval()
with torch.no_grad():
    out = model.forward(
        pixel_values=data["pixel_values"].unsqueeze(0).repeat(4,1,1,1).cuda(),
        masks=data["masks"].unsqueeze(0).repeat(4,1,1).cuda(),
        organ_id=torch.Tensor([-1]).long().repeat(4).cuda(),
    )
out['loss']

In [ ]:
enc = model.encode(data['pixel_values'].unsqueeze(0).cuda(), torch.Tensor([data['organ_id']]).long().cuda())[0]

In [ ]:
from segment_anything import sam_model_registry
from copy import deepcopy
from nets.segm_net import UNet2DFiLM, MedSAM, MedSAMPrompt
from safetensors.torch import load_file
sam_model = sam_model_registry["vit_b"](checkpoint=MEDSAM_BASE_WEIGHTS)

model = MedSAM(
    image_encoder=deepcopy(sam_model.image_encoder),
    mask_decoder=deepcopy(sam_model.mask_decoder),
    prompt_encoder=deepcopy(sam_model.prompt_encoder),
    predict_bboxes=True,
    freeze_image_encoder=0,
)
state_dict = load_file(
    "/work/tesi_nmorelli/UUSIC_new/checkpoints/medsam_unfreezed/model.safetensors"
)
model.load_state_dict(state_dict)
load_result = model.load_state_dict(state_dict)
print(load_result)
model.eval()
model = model.cuda()

In [ ]:
with torch.no_grad():
    image_embedding = model.image_encoder(data['pixel_values'].unsqueeze(0).cuda()) 
    image_embedding

In [ ]:
with torch.no_grad():
    image_pe = model.prompt_encoder.get_dense_pe()
            # Decode mask
    low_res_masks, iou_predictions = model.mask_decoder(
            image_embeddings=image_embedding,  # (B, 256, 64, 64)
            image_pe=image_pe,  # (1, 256, 64, 64)
            sparse_prompt_embeddings=model.learned_sparse_embeddings,  # (B, 2, 256)
            dense_prompt_embeddings=model.learned_dense_embeddings,  # (B, 256, 64, 64)
            multimask_output=False,
        ) 

In [8]:
from transformers import Trainer
from torch.utils.data import DataLoader

class CustomTrainerWithSampler(Trainer):
    """
    Custom Trainer that uses BalancedHierarchicalSampler for training.
    """
    
    def __init__(self, *args, train_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_sampler = train_sampler
    
    def get_train_dataloader(self):
        """
        Override to use our custom batch sampler.
        """
        if self.train_sampler is None:
            # Fallback to default behavior
            return super().get_train_dataloader()
        
        # Create DataLoader with our batch sampler
        return DataLoader(
            self.train_dataset,
            batch_sampler=self.train_sampler,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
            persistent_workers=self.args.dataloader_persistent_workers,
            prefetch_factor=self.args.dataloader_prefetch_factor if self.args.dataloader_num_workers > 0 else None,
        )

In [9]:
from train.segm_train import compute_metrics

trainer = CustomTrainerWithSampler(
        model=model,
        args=training_args,
        train_dataset=dataset,
        eval_dataset=dataset,
        compute_metrics=compute_metrics,
        train_sampler=train_sampler,
    )

max_steps is given, it will override any value given in num_train_epochs


In [10]:
metrics = trainer.evaluate()

/homes/nmorelli/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(

***** Running Evaluation *****
  Num examples = 1163
  Batch size = 8


Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"
wandb: Currently logged in as: nmorelli (challenge) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Could not log the number of model parameters in Weights & Biases due to an AttributeError.


In [ ]:
metrics

In [ ]:
data = dataset.__getitem__(0)
data.keys()
out = model(pixel_values = data['pixel_values'].unsqueeze(0).cuda(), organ_id = data['organ_id'].clone().cuda())

In [ ]:
out

In [ ]:
from torchvision.transforms.v2.functional import to_pil_image

to_pil_image((torch.sigmoid(out['logits']) > 0.7).to(torch.float32))